# Correlation


> Part of *PHYS 0241: Biomedical Imaging* (Prof. Michael E. Durst, Middlebury College). The [textbook page for this notebook](https://durstlab.middcreate.net/teaching/protected/textbook-biomedicalimaging/python/ch05/ch05-21.html) shows the rendered version together with the pre-rendered animations (course password required).
>
> **Start here:** no setup is needed for this notebook. In Google Colab, choose **File > Save a copy in Drive** to keep your changes.
>
> *Provided for academic use in PHYS 0241; please do not repost or redistribute ([terms of use](https://github.com/durstme/textbook-biomedicalimaging-notebooks/blob/main/LICENSE.md)).*

Companion to the **Correlation** section of Chapter 5.

Every claim that section makes is checked here numerically: that
convolution flips and correlation does not, that the autocorrelation of a
rect is a triangle, that the transform of an autocorrelation is the power
spectrum, and that the incoherent OTF really is the pupil autocorrelated
with itself.


## Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, util

plot_params = {
    'font.size': 12,
    'font.family': 'sans-serif',
    'mathtext.fontset': 'stix',
    'axes.linewidth': 1.0,
    'lines.linewidth': 2.0,
    'figure.autolayout': True,
}
plt.rcParams.update(plot_params)


def apply_custom_formatting(ax):
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.minorticks_on()
    ax.tick_params(which='both', direction="in", width=1)
    ax.tick_params(which='major', length=7)
    ax.tick_params(which='minor', length=3.5)
    ax.xaxis.set_ticks_position('both')
    ax.yaxis.set_ticks_position('both')

## 1. The one difference: convolution flips, correlation does not

Both are computed here with the FFT, using the two theorems:

$$\mathscr{F}\{f \otimes h\} = F(k)\,H(k) \qquad\qquad \mathscr{F}\{f \star h\} = F^*(k)\,H(k)$$

The **only** difference in the code is the `np.conj`.

In [ ]:
# --- 1. Grid ---
n = 1024
x = np.linspace(-10, 10, n)
dx = x[1] - x[0]

# The DFT's own coordinate ordering (0, dx, 2dx, ..., -2dx, -dx).
# Building a symmetric function on THIS grid makes it symmetric under the
# flip the DFT actually performs, so the comparison below is exact rather
# than off by one sample.
xf = np.fft.fftfreq(n) * n * dx


def correlate(f, h):
    """Cross-correlation via the correlation theorem: conj(F) . H, transformed back."""
    return np.fft.fftshift(np.fft.ifft(np.conj(np.fft.fft(f)) * np.fft.fft(h)).real) * dx


def convolve_fft(f, h):
    """Convolution via the convolution theorem: F . H, transformed back."""
    return np.fft.fftshift(np.fft.ifft(np.fft.fft(f) * np.fft.fft(h)).real) * dx

Use a deliberately **asymmetric** kernel, because with a symmetric one the two
operations give identical answers and the distinction is invisible.

In [ ]:
# --- 2. An asymmetric kernel: a one-sided exponential decay ---
signal = ((x > -6) & (x < -3)).astype(float)                  # a rectangle
kernel = np.where((xf >= 0) & (xf < 4), np.exp(-np.clip(xf, 0, None)), 0.0)

corr = correlate(kernel, signal)
conv = convolve_fft(kernel, signal)

fig1, axes = plt.subplots(1, 3, figsize=(13, 3.2))
axes[0].plot(x, signal)
axes[0].set_title('Signal $f(x)$')
axes[1].plot(np.fft.fftshift(xf), np.fft.fftshift(kernel), color='tab:orange')
axes[1].set_title('Kernel $h(x)$ (asymmetric)')
axes[2].plot(x, conv, label=r'convolution $f \otimes h$')
axes[2].plot(x, corr, '--', label=r'correlation $f \star h$')
axes[2].set_title('Flipped in one, not the other')
axes[2].legend()
for a in axes:
    a.set_xlabel('$x$')
apply_custom_formatting(axes[2])
# fig1.savefig("../figures/ch05/ch05-21-correlation-flip.jpg", dpi=200, bbox_inches='tight')
plt.show()

print('asymmetric kernel: max |correlation - convolution| = %.4f' % np.abs(corr - conv).max())

The two results are **mirror images** of each other.

Now repeat with a symmetric kernel and the difference vanishes to machine precision.
This is why the distinction so rarely matters in imaging: the point spread function of a
well-aligned lens is symmetric, so blurring by it is the same operation either way.

In [ ]:
# --- 3. Symmetric kernel: the two operations coincide ---
kernel_sym = np.exp(-xf**2)
d = np.abs(correlate(kernel_sym, signal) - convolve_fft(kernel_sym, signal)).max()
print('symmetric kernel:  max |correlation - convolution| = %.2e' % d)

## 2. Autocorrelation of a rectangle is a triangle

This reproduces the worked example in the chapter. The autocorrelation at displacement
$X$ is the **overlap area** of the rectangle with a copy of itself shifted by $X$, so it
starts at $d$ and falls linearly to zero once the shift exceeds the full width.

In [ ]:
# --- 1. Autocorrelate a rectangle of width d ---
width = 4.0
rect = (np.abs(xf) <= width / 2).astype(float)
auto = correlate(rect, rect)
tri = np.where(np.abs(x) <= width, width - np.abs(x), 0.0)      # predicted d*tri(X/d)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].plot(np.fft.fftshift(xf), np.fft.fftshift(rect))
axes[0].set_title('rect$(x/d)$,  $d=%.0f$' % width)
axes[1].plot(x, auto, label='autocorrelation')
axes[1].plot(x, tri, '--', label=r'$d\,$tri$(X/d)$')
axes[1].set_title('Autocorrelation is a triangle')
axes[1].legend()
plt.show()

print('peak = %.3f   (expected d = %.1f)' % (auto.max(), width))
print('max |autocorrelation - d*tri| = %.1e  (about one grid sample, dx = %.1e)'
      % (np.abs(auto - tri).max(), dx))

The residual is about one grid spacing, which is what sampling a discontinuous
rectangle on a finite grid costs. The identity in the next section, which involves no
discontinuity in the comparison, holds to machine precision.

## 3. The autocorrelation theorem: the transform is the power spectrum

$$\mathscr{F}\{f \star f\} = F^*(k) F(k) = |F(k)|^2$$

The phase disappears, so the autocorrelation keeps the frequency content of $f$ and none
of its position information.

In [ ]:
# --- 1. Transform the autocorrelation and compare with |F|^2 ---
k = 2 * np.pi * np.fft.fftfreq(n, d=dx)
F = np.fft.fft(rect) * dx
AC = np.fft.fft(np.fft.ifftshift(auto)) * dx

print('max |F{autocorrelation} - |F|^2| = %.2e' % np.abs(AC - np.abs(F)**2).max())

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(np.fft.fftshift(k), np.fft.fftshift(np.abs(F)**2), label='$|F(k)|^2$')
ax.plot(np.fft.fftshift(k), np.fft.fftshift(AC.real), '--',
        label=r'$\mathscr{F}\{f \star f\}$')
ax.set_xlim(-15, 15)
ax.set_xlabel('$k$')
ax.legend()
ax.set_title('Transform of the autocorrelation is the power spectrum')
plt.show()

In [ ]:
# --- 2. Shift invariance: a displaced object has the SAME autocorrelation ---
rect_shifted = np.roll(rect, 150)
print('max |autocorr(rect) - autocorr(shifted rect)| = %.2e'
      % np.abs(correlate(rect, rect) - correlate(rect_shifted, rect_shifted)).max())
print('\nTwo objects differing only by a shift are indistinguishable by autocorrelation.')

## 4. Correlation finds a known shape in an image

This is the practical use, and it is exactly the microscope problem: locate point-like
sources (beads, fluorophores, fiducial markers) in a noisy field. Correlating the image
with a template of one bead produces a peak at every bead.

In [ ]:
# --- 1. A synthetic field of beads, badly buried in noise ---
rng = np.random.default_rng(3)
M = 256
yy, xx = np.mgrid[0:M, 0:M]
centers = [(40, 60), (120, 180), (200, 90), (70, 200), (170, 40)]
sigma_bead = 3.0

beads = np.zeros((M, M))
for cy, cx in centers:
    beads += np.exp(-((yy - cy)**2 + (xx - cx)**2) / (2 * sigma_bead**2))
noisy = beads + 0.35 * rng.standard_normal((M, M))

# The template is one bead, the same shape the imaging system produces: the PSF.
template = np.exp(-((yy - M // 2)**2 + (xx - M // 2)**2) / (2 * sigma_bead**2))

In [ ]:
# --- 2. Correlate, using the correlation theorem in 2D ---
corr2d = np.fft.fftshift(
    np.fft.ifft2(np.conj(np.fft.fft2(template)) * np.fft.fft2(noisy)).real)

# Pull out the five strongest peaks
found, scratch = [], corr2d.copy()
for _ in range(len(centers)):
    p = np.unravel_index(np.argmax(scratch), scratch.shape)
    found.append(p)
    scratch[max(0, p[0]-8):p[0]+8, max(0, p[1]-8):p[1]+8] = -np.inf

print('true bead centers   :', sorted(centers))
print('correlation maxima  :', sorted(tuple(int(v) for v in p) for p in found))

In [ ]:
# --- 3. Show the result ---
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(noisy, cmap='gray')
axes[0].set_title('Noisy image (can you see 5 beads?)')
axes[1].imshow(template, cmap='gray')
axes[1].set_title('Template (one bead = the PSF)')
axes[2].imshow(corr2d, cmap='viridis')
axes[2].set_title('Correlation: a peak at every bead')
for p in found:
    axes[2].plot(p[1], p[0], 'r+', markersize=13, markeredgewidth=2)
for a in axes:
    a.axis('off')
plt.show()

Correlating with the PSF is the optimal way to find a point source in noise, which is
why it is what particle-tracking and single-molecule localization software actually does.

**A caution worth knowing.** This plain, unnormalized correlation works here because the
background is near zero. On an ordinary photograph it fails: the correlation is dominated
by whichever region is brightest rather than whichever best matches the template. Real
template matching therefore uses *normalized* cross-correlation, for example
`skimage.feature.match_template`.

## 5. The optical transfer function is an autocorrelation

The incoherent OTF is the autocorrelation of the **pupil function**, normalized so that
$\mathrm{OTF}(0,0)=1$. Physically: the OTF at a given spatial frequency is the overlap
area of the aperture with a displaced copy of itself, which is why it falls to zero at a
hard cutoff.

The cell below checks this against the route used in the chapter,
$\mathrm{OTF} = \mathscr{F}\{h^2\}$.

In [ ]:
# --- 1. A circular pupil, its incoherent PSF, and the OTF computed both ways ---
m = 256
fx = np.fft.fftshift(np.fft.fftfreq(m))
KX, KY = np.meshgrid(fx, fx)
pupil = ((KX**2 + KY**2) <= 0.15**2).astype(float)

# The incoherent PSF is |FT{pupil}|^2
psf = np.abs(np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(pupil))))**2

# Route A, as in the chapter: OTF = F{h^2}
otf_from_psf = np.abs(np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(psf))))
otf_from_psf /= otf_from_psf.max()

# Route B: OTF = autocorrelation of the pupil
otf_from_pupil = np.fft.fftshift(
    np.fft.ifft2(np.abs(np.fft.fft2(np.fft.ifftshift(pupil)))**2).real)
otf_from_pupil = np.abs(otf_from_pupil) / np.abs(otf_from_pupil).max()

print('max |F{h^2} - normalized pupil autocorrelation| = %.2e'
      % np.abs(otf_from_psf - otf_from_pupil).max())
print('The two routes are the same calculation.')

In [ ]:
# --- 2. Compare the MTF profile with the analytic circular-aperture result ---
profile = otf_from_pupil[m // 2, m // 2:]
kc = 2 * 0.15 * m                          # cutoff: twice the pupil radius
s = np.clip(np.arange(len(profile)) / kc, 0, 1)
analytic = (2 / np.pi) * (np.arccos(s) - s * np.sqrt(np.clip(1 - s**2, 0, None)))

fig5, axes = plt.subplots(1, 3, figsize=(13, 3.4))
axes[0].imshow(pupil, cmap='gray')
axes[0].set_title('Pupil function $P$')
axes[0].axis('off')
axes[1].imshow(otf_from_pupil, cmap='viridis')
axes[1].set_title(r'OTF $= P \star P$')
axes[1].axis('off')
axes[2].plot(profile, label='computed')
axes[2].plot(analytic, '--', label='analytic')
axes[2].axvline(kc, color='gray', ls=':', label='cutoff')
axes[2].set_xlim(0, kc * 1.3)
axes[2].set_xlabel('spatial frequency (pixels)')
axes[2].set_title('MTF')
axes[2].legend()
apply_custom_formatting(axes[2])
# fig5.savefig("../figures/ch05/ch05-21-otf-autocorrelation.jpg", dpi=200, bbox_inches='tight')
plt.show()

print('max |computed MTF - analytic| = %.3f' % np.abs(profile - analytic).max())

The analytic incoherent MTF of a circular aperture is

$$\mathrm{MTF}(s) = \frac{2}{\pi}\left[\arccos s - s\sqrt{1-s^2}\right],
\qquad s = k/k_c$$

and it reaches zero at the cutoff because the two displaced copies of the aperture stop
overlapping. A **slit** aperture autocorrelates to a triangle, so its MTF falls linearly.
That is the same rectangle-into-triangle result as Section 2.

## 6. Interactive: the aperture sets the cutoff, and the cutoff sets the blur

Shrink the pupil and the autocorrelation narrows, the cutoff frequency drops, and the
image loses fine detail. Blur is not something added to an image. It is what is left
after the aperture removes the high spatial frequencies.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

obj = util.img_as_float(data.camera())
M2 = obj.shape[0]
FX = np.fft.fftshift(np.fft.fftfreq(M2))
AX, AY = np.meshgrid(FX, FX)
RHO = np.sqrt(AX**2 + AY**2)
OBJ_FT = np.fft.fft2(obj)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
im0 = axes[0].imshow(np.zeros((M2, M2)), cmap='gray')
axes[0].set_title('Pupil')
im1 = axes[1].imshow(np.zeros((M2, M2)), cmap='viridis')
axes[1].set_title(r'OTF $= P \star P$')
im2 = axes[2].imshow(obj, cmap='gray', vmin=0, vmax=1)
axes[2].set_title('Image')
for a in axes:
    a.axis('off')
plt.close(fig)


def update(radius):
    P = (RHO <= radius).astype(float)
    otf = np.fft.fftshift(np.fft.ifft2(np.abs(np.fft.fft2(np.fft.ifftshift(P)))**2).real)
    otf = np.abs(otf)
    otf /= otf.max()
    img = np.fft.ifft2(OBJ_FT * np.fft.ifftshift(otf)).real
    im0.set_data(P)
    im1.set_data(otf)
    im2.set_data(np.clip(img, 0, 1))
    axes[2].set_title('Image (pupil radius %.3f)' % radius)
    display(fig)


widgets.interact(update, radius=widgets.FloatSlider(
    value=0.12, min=0.02, max=0.45, step=0.01, description='pupil radius'));